# 02 · Your first model call: messages, usage, cost and streaming
Level L1 · Part 1 · core · ~20 min · builds on 01

## Goal
Make a model call with the official SDK, read the usage and cost of each call, and stream the answer.

## Why it matters
Without an SDK, a model call is a plain HTTP request. You build the URL and the JSON body by hand,
add the key header, dig the reply out of nested JSON, count **tokens** from another field (a token is
a small chunk of text, about 3/4 of a word; lesson 03 goes deeper), and write your own retry loop for
the 429 and 503 errors that every busy API returns (429 = too many requests, 503 = server busy).
Here is the body you would build by hand. **Nothing is sent in this cell.**

In [1]:
import json

hand_built_request = {
    "systemInstruction": {"parts": [{"text": "You are the Brightside Outfitters support desk."}]},
    "contents": [{"role": "user", "parts": [{"text": "Parcel 1093 never came. Refund?"}]}],
    "generationConfig": {"temperature": 0, "maxOutputTokens": 200},
}
print(json.dumps(hand_built_request, separators=(",", ":")))
print("\nReply text would be at: candidates[0].content.parts[0].text")

{"systemInstruction":{"parts":[{"text":"You are the Brightside Outfitters support desk."}]},"contents":[{"role":"user","parts":[{"text":"Parcel 1093 never came. Refund?"}]}],"generationConfig":{"temperature":0,"maxOutputTokens":200}}

Reply text would be at: candidates[0].content.parts[0].text


The body above uses the REST camelCase names (`systemInstruction`, `generationConfig`). The SDK takes
the snake_case arguments `system_instruction` and `config`, so you never write this JSON yourself.

**What the SDK makes easy:** one call takes the messages and settings, returns the parsed reply and
the usage, and retries are a few lines you can add. **When you don't need it:** a one-off script
that needs a single reply can skip the SDK, but a real app ends up needing these parts anyway.

**Set up the client once.** The key is read from `.env` (never typed here). The first run with a key
makes real calls and saves them in `data/llm_cassettes/`; later runs replay the saved answers, so no key
is needed. To record again after changing a prompt, run with `TUT_LLM_MODE=record` and a key in `.env`.

In [2]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("02_first-model-call")
from google import genai  # the official SDK; in your own project: genai.Client(api_key=...)
from google.genai import errors, types
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

# One pinned model id for the whole course (RESEARCH.md section 1). Price in USD per 1M tokens (checked 2026-10-20).
MODEL = "gemini-3.1-flash-lite"
PRICE_IN, PRICE_OUT = 0.25, 1.50
RETRY_INITIAL, RETRY_MAX, RETRY_JITTER, TIMEOUT_S = 5.0, 30.0, 5.0, 30

NO_KEY = "replay"  # while replaying, the recording answers and this placeholder is never sent
client = genai.Client(
    vertexai=True,
    api_key=NO_KEY if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    # attempts=1: the SDK makes one try per request; tenacity below does the retrying, so you can see it.
    http_options=types.HttpOptions(timeout=TIMEOUT_S * 1000, retry_options=types.HttpRetryOptions(attempts=1)),
)
print("Client ready. Replaying a recording:", llm_replay.is_replaying())

LLM mode: replay (02_first-model-call.yaml)


Client ready. Replaying a recording: True


**Retries.** A 429 (too many requests) or a 5xx (the server failed or is busy, such as 503) is a
signal to try again later. `ask()` below wraps one call with tenacity. `@retry` re-runs the
function when it raises an error, and `is_transient` decides which errors are worth a retry.
How long to wait between tries, and why the waits get random jitter, is explained in lesson 32.

In [3]:
TRANSIENT = {408, 429, 500, 502, 503, 504}


def is_transient(e: BaseException) -> bool:
    """True for API errors whose code means 'try again later'."""
    return isinstance(e, errors.APIError) and e.code in TRANSIENT


@retry(retry=retry_if_exception(is_transient),
       stop=stop_after_attempt(4),
       wait=wait_exponential_jitter(initial=RETRY_INITIAL, max=RETRY_MAX, jitter=RETRY_JITTER),
       reraise=True)  # after the 4th failed try, raise the original API error, not tenacity's wrapper
def ask(**kwargs):
    """One model call, retried on transient errors. Each retry is a request too."""
    return client.models.generate_content(**kwargs)

**The retry limits (no call is made here).** `stop_after_attempt(4)` means 4 attempts = 3 retries. The waits
between tries (backoff and jitter) are set by `wait_exponential_jitter`; lesson 32 works them out.
Each request also gives up after 30 s (the timeout). A 429 or 5xx is retried. A 400 (bad request) or
401 (bad key) is not, because waiting cannot fix it.

## The idea
A model call takes a **list of messages**. Each message has a **role**: `user` for the customer,
`model` for the assistant's earlier reply (Gemini's name for it; not `assistant`). The instructions go
in `system_instruction`, not in the list. The call returns one reply and a **usage** record: how many
input tokens you sent and how many output tokens came back. You pay for both. The API keeps **no memory**
between calls, so to continue a conversation your code sends the whole history again.

Analogy: a phone call to a helpdesk where the agent forgets everything when you hang up. Each time
you call back, you have to repeat the story.

## Picture
```mermaid
sequenceDiagram
  participant App as Your code
  participant API as Model API
  App->>API: system instruction + full message list
  API-->>App: reply + usage (input and output tokens)
  App->>API: next turn: the whole list again, plus the new message
  API-->>App: reply + usage (input grows every turn)
```

## Step by step
**Step 1 · one call.** The support desk gets a message. Ask once, with a system instruction and
`temperature=0`. If your quota is used up, the API answers 429 and `ask()` waits and tries again.

In [4]:
SYSTEM = ("You are the Brightside Outfitters support desk. You cannot see order data. "
          "Say what you can and cannot check. Answer in two sentences.")
CONFIG = types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0, max_output_tokens=200)
question = "My parcel for order 1093 never came. Can I get my money back?"

resp = ask(model=MODEL, contents=[{"role": "user", "parts": [{"text": question}]}], config=CONFIG)
print("Reply:", resp.text)
print("Finish reason:", resp.candidates[0].finish_reason.name)

Reply: I cannot access your personal order details or tracking information, so I am unable to verify the status of your shipment. Please contact our billing department with your order number so they can review your eligibility for a refund.
Finish reason: STOP


**Step 2 · read the usage and the cost.** Every reply carries `usage_metadata` with four counts:
`prompt_token_count` (input), `candidates_token_count` (the reply), `thoughts_token_count` (**thinking
tokens**: the model's hidden reasoning, billed like output; 0 if the model does not think) and
`total_token_count`. Cost = input tokens × input price + (reply + thinking tokens) × output price.
The function below only reads numbers the call already returned.

In [5]:
def show_usage(response) -> float:
    u = response.usage_metadata
    out_tokens = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
    cost = (u.prompt_token_count * PRICE_IN + out_tokens * PRICE_OUT) / 1_000_000
    print(f"input {u.prompt_token_count} tokens · output {out_tokens} (incl. thinking) · total {u.total_token_count}")
    print(f"cost of this call: ${cost:.6f}")
    return cost

_ = show_usage(resp)

input 47 tokens · output 42 (incl. thinking) · total 89
cost of this call: $0.000075


**Step 3 · statelessness.** The customer asks a follow-up. Ask it alone first, as a new call. Then
send the history. Build the list as the conversation goes: after each call, append the turns you just made.

In [6]:
follow_up = "What was the order number I just gave you? Answer with the number only."

alone = ask(model=MODEL, contents=[{"role": "user", "parts": [{"text": follow_up}]}], config=CONFIG)
print("Without history:", alone.text)

history = []
history.append({"role": "user", "parts": [{"text": question}]})      # turn 1: the customer's question
history.append({"role": "model", "parts": [{"text": resp.text}]})    # turn 1: the reply from Step 1
history.append({"role": "user", "parts": [{"text": follow_up}]})     # turn 2: the follow-up
together = ask(model=MODEL, contents=history, config=CONFIG)
history.append({"role": "model", "parts": [{"text": together.text}]})  # the list now holds the whole conversation
print("With history:   ", together.text)
_ = show_usage(together)

Without history: I do not have access to your personal order history or any information you may have previously provided. I can assist with general store policies and product inquiries, but I cannot view or verify specific order numbers.
With history:    1093
input 105 tokens · output 4 (incl. thinking) · total 109
cost of this call: $0.000032


Compare the input token counts of the two calls. The call "with history" resends the earlier question
and reply, so its input is bigger. That growth is the price of memory.

**Step 4 · streaming.** `generate_content_stream` yields the reply in chunks while the model writes
it, so the customer sees the first words early. Chunks break where the model's tokens end, so a word
can be split across two chunks: print each one with `end=""` and `flush=True`. The total cost does not
change. Streams are not retried here: a retry halfway through would repeat words the customer already saw.

In [7]:
stream = client.models.generate_content_stream(
    model=MODEL,
    contents=[{"role": "user", "parts": [{"text": "A parcel is lost in transit. In two sentences, what can the customer do next?"}]}],
    config=CONFIG,
)
chunks, with_usage = 0, None
for chunk in stream:
    chunks += 1
    if chunk.text:
        print(chunk.text, end="", flush=True)
    if chunk.usage_metadata:  # the usage record arrives on the last chunk
        with_usage = chunk
print()
print("Chunks received:", chunks)
_ = show_usage(with_usage)

I cannot

 access your order history or track specific shipments, but I can provide general guidance on how to contact

 the carrier. Please reach out to our support team with your order number so we can initiate a formal investigation into your missing

 parcel.


Chunks received: 5
input 46 tokens · output 47 (incl. thinking) · total 93
cost of this call: $0.000082


**Save the recording.** The four calls above (Steps 1 to 4) are now in `data/llm_cassettes/`, so they
replay with no key.

In [8]:
llm_replay.stop()

## At work
- Teams log usage and cost on every call, and cap the spend per request and per day. Cache-aware
  counts change the bill → lesson 33 (later in the course).
- Rate limits and 503s are normal in production. Teams retry with backoff and jitter, honour
  `Retry-After`, and set timeouts, so one slow call does not hold a worker → lesson 32 (later in the course).
- Resending history is the main cost of a long chat: every turn pays for all the earlier turns again →
  lesson 03 (context windows and what fills them).

## Common mistakes
1. **Sending the follow-up alone.** The model answers as if the earlier turns never happened, because
   they were never sent. Resend the history, as in Step 3.
2. **Writing `assistant` for the model's turns.** Per the SDK docs, Gemini uses `model`. Use the role the API expects.
3. **Assuming `text` is always there.** Per the SDK docs, a reply cut off by `max_output_tokens`, or blocked,
   can come back with no text. Check `finish_reason` first.
4. **Passing a value of the wrong type.** The SDK's config is a pydantic model (a Python library that checks
   data types). A wrong type raises pydantic's `ValidationError` before anything leaves your machine.
   The cell below shows the message.

In [9]:
types.GenerateContentConfig(max_output_tokens="a short answer")

ValidationError: 1 validation error for GenerateContentConfig
max_output_tokens
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='a short answer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing

## Try it
Your turn: the desk expects 2,000 messages a day, each like the first call above. The cost of one call is
input tokens × `PRICE_IN` + (reply + thinking tokens) × `PRICE_OUT`, all divided by 1,000,000. Using the
usage of `resp` from Step 1 as the average, estimate the cost for a 30-day month. Write the calculation
with the variables `resp`, `PRICE_IN` and `PRICE_OUT`.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
# Uses the `resp` from Step 1.
u = resp.usage_metadata
out_tokens = (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
per_call = (u.prompt_token_count * PRICE_IN + out_tokens * PRICE_OUT) / 1_000_000
print(f"per call ${per_call:.6f} · per 30-day month ${per_call * 2000 * 30:,.2f}")
# Real traffic is bigger: follow-up turns resend history, so the average input is higher.

per call $0.000075 · per 30-day month $4.48


## Recap
- **Message, role, system instruction.** A call sends a list of messages with roles (`user`, `model`);
  the **system instruction** holds the rules and is sent apart.
- **Usage.** Each reply counts `prompt_token_count` (input), `candidates_token_count` (reply) and
  `thoughts_token_count` (**thinking tokens**: hidden reasoning, billed like output). A **token** is a
  small chunk of text, about 3/4 of a word; lesson 03 goes deeper.
- **Stateless API.** The server keeps no memory, so your code resends the history. **Streaming** shows
  the reply as it is written and does not change the cost.
- **Retries and timeouts.** 429 and 5xx are retried (backoff and jitter: lesson 32), each request has a
  30 s **timeout**; 400 and 401 are not retried.

**Interview:** "A chat app's bill doubled after a month. Where do you look?"
Strong answer: "At usage per turn. Input grows each turn because the history is resent, so I'd cap or
summarise it. Then I'd check retries, since they bill again."